In [1]:
import os, random
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

INPUT_CSV = "/Users/suyashmali/Case_Study_Materials/LLM/Notebook/Cleaned_Data_1.csv"
TEXT_COL = "metadata_clean"
LABEL_COL = "Level3"

MIN_SAMPLES = 100
MAX_SAMPLES = 500

EMBED_MODEL = "all-mpnet-base-v2"

/opt/anaconda3/envs/taxonomy_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
df = pd.read_csv(INPUT_CSV)

def min_max_sample(df, label_col, min_k, max_k, seed=42):
    sampled_groups = []
    for label, g in df.groupby(label_col):
        if len(g) < min_k:
            continue
        n = min(len(g), max_k)
        sampled_groups.append(g.sample(n=n, random_state=seed))
    return pd.concat(sampled_groups).reset_index(drop=True)

sample_df = min_max_sample(df, LABEL_COL, MIN_SAMPLES, MAX_SAMPLES)

print("Final sample size:", len(sample_df))
print("Unique Level3 categories:", sample_df[LABEL_COL].nunique())

print(sample_df[[LABEL_COL]].value_counts().head(25))

Final sample size: 43803
Unique Level3 categories: 126
Level3                         
Wireless Routers                   500
Power Banks                        500
Network Switches                   500
Projector Mounts                   500
Projector Lamps                    500
Household Batteries                500
IT Support Services                500
Projector Accessories              500
Projection Screens                 500
Power Supply Units                 500
Power Distribution Units (PDUs)    500
Power Cables                       500
KVM Switches                       500
Keyboards                          500
Power Adapters & Inverters         500
Flat Panel Spare Parts             500
Portable Speakers                  500
PCs/Workstations                   500
Mice                               500
Mobile Device Chargers             500
PC/Workstation Barebones           500
Mobile Device Keyboards            500
Mobile Headsets                    500
Notebooks       

In [3]:
sample_df.columns

Index(['Brand', 'BrandPartCode', 'ProductName', 'Description.LongProductName',
       'Description.LongDesc', 'SummaryDescription.LongSummaryDescription',
       'pathlist_names', 'Level1', 'Level2', 'Level3', 'metadata_text',
       'metadata_clean'],
      dtype='object')

In [4]:
# sample_df[sample_df["ProductName"] == "C30"][["metadata_clean", "pathlist_names"]]

In [5]:
print(sample_df[[LABEL_COL]].value_counts().head(130))

Level3                        
Wireless Routers                  500
Power Banks                       500
Network Switches                  500
Projector Mounts                  500
Projector Lamps                   500
                                 ... 
Portable Stereo Systems           108
Composite Video Cables            106
Component (YPbPr) Video Cables    104
Mobile Phone Cables               103
Input Device Accessories          100
Name: count, Length: 126, dtype: int64


In [6]:
sample_df["final_text"] = (
    sample_df[TEXT_COL]
    .fillna("")
    .astype(str)
    .str.strip()
)

fallback_cols = ["ProductName", "Title", "SummaryDescription.LongSummaryDescription"]

def fallback_text(row):
    if row["final_text"]:
        return row["final_text"]
    return " ".join(
        str(row[c]) for c in fallback_cols
        if c in row and pd.notna(row[c])
    )

sample_df["final_text"] = sample_df.apply(fallback_text, axis=1)

print("Non-empty texts:", sample_df["final_text"].astype(bool).sum())

Non-empty texts: 43803


In [7]:
sample_df[TEXT_COL].head(1)

0    vertiv ems1000t, avocent. type av transmitter,...
Name: metadata_clean, dtype: object

In [8]:
sample_df["final_text"].head(1)

0    vertiv ems1000t, avocent. type av transmitter,...
Name: final_text, dtype: object

In [9]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("max_colwidth", None)

In [11]:
model = SentenceTransformer(EMBED_MODEL)

embeddings = model.encode(
    sample_df["final_text"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/343 [00:00<?, ?it/s]

In [11]:
umap_model = UMAP(
    n_neighbors=15,
    min_dist=0.0,
    n_components=25,
    metric="cosine",
    random_state=RANDOM_STATE
)

reduced = umap_model.fit_transform(embeddings)
print("Reduced shape:", reduced.shape)

/opt/anaconda3/envs/taxonomy_env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Reduced shape: (43803, 25)


In [13]:
# from sklearn.cluster import AgglomerativeClustering
# from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# thresholds = [0.5, 0.55, 0.6]

# results = []

# for t in thresholds:
#     agg = AgglomerativeClustering(
#         n_clusters=None,
#         distance_threshold=t,
#         linkage="average",
#         metric="cosine"
#     )
    
#     labels = agg.fit_predict(embeddings)
    
#     n_clusters = len(set(labels))
#     ari = adjusted_rand_score(sample_df[LABEL_COL], labels)
#     nmi = normalized_mutual_info_score(sample_df[LABEL_COL], labels)
    
#     results.append((t, n_clusters, ari, nmi))
    
#     print(
#         f"threshold={t:<5} | clusters={n_clusters:<4} | "
#         f"ARI={ari:.4f} | NMI={nmi:.4f}"
#     )

In [20]:
# sample_df.head()


In [12]:
sample_df['C_id_agg'] = labels  # from threshold=0.5
sample_df['C_id_agg'].value_counts().describe()

count     209.000000
mean      209.583732
std       577.307272
min         1.000000
25%         2.000000
50%         7.000000
75%        67.000000
max      5099.000000
Name: count, dtype: float64

In [13]:
# sample_df['C_id_agg'].nunique()

In [14]:
# sample_df[sample_df["C_id_agg"] == example_cluster][["Brand", "ProductName", LABEL_COL]].head(10)

In [15]:
sample_df.columns

Index(['Brand', 'BrandPartCode', 'ProductName', 'Description.LongProductName',
       'Description.LongDesc', 'SummaryDescription.LongSummaryDescription',
       'pathlist_names', 'Level1', 'Level2', 'Level3', 'metadata_text',
       'metadata_clean', 'final_text', 'C_id_agg'],
      dtype='object')

In [16]:
# cluster = sample_df['C_id_agg'].value_counts().index[99] 
# print("Inspecting HDBSCAN cluster:", cluster)
# display_cols = ['Brand', 'ProductName', 'metadata_clean', 'Level3', 'C_id_agg']
# display(sample_df[sample_df['C_id_agg'] == cluster][display_cols].head(100))

In [17]:
sample_df[sample_df['C_id_agg'] == 45][['Brand', 'ProductName', 'metadata_clean', 'Level3', 'C_id_agg']].nunique()

Brand             2
ProductName       4
metadata_clean    4
Level3            2
C_id_agg          1
dtype: int64

In [14]:
agg = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.5,
    linkage="average",
    metric="cosine"
)

sample_df["C_id_agg"] = agg.fit_predict(embeddings)

print("Clusters formed:", sample_df["C_id_agg"].nunique())
print(sample_df["C_id_agg"].value_counts().head(10))

In [42]:
ari = adjusted_rand_score(sample_df[LABEL_COL], sample_df["C_id_agg"])
nmi = normalized_mutual_info_score(sample_df[LABEL_COL], sample_df["C_id_agg"])

print("ARI:", round(ari, 4))
print("NMI:", round(nmi, 4))

ARI: 0.3777
NMI: 0.8024


In [19]:
def cluster_purity(df, cluster_col, gold_col):
    stats = (
        df.groupby(cluster_col)[gold_col]
        .value_counts()
        .rename("count")
        .reset_index()
    )
    top = stats.groupby(cluster_col).first().reset_index()
    sizes = df[cluster_col].value_counts().rename("size").reset_index()
    merged = top.merge(sizes, on=cluster_col)
    merged["purity"] = merged["count"] / merged["size"]
    return merged.sort_values("purity", ascending=False)

purity_df = cluster_purity(sample_df, "C_id_agg", LABEL_COL)
purity_df.head(10)

,C_id_agg,Level3,count,size,purity
208,208,Network Switch Components,1,1,1.0
167,167,E-Book Reader Cases,1,1,1.0
111,111,Software Licenses/Upgrades,1,1,1.0
108,108,Patch Panels,1,1,1.0
107,107,Battery Chargers,1,1,1.0
106,106,Wired Routers,1,1,1.0
161,161,Calculators,1,1,1.0
1,1,Software Licenses/Upgrades,6,6,1.0
101,101,InfiniBand Cables,5,5,1.0
162,162,Servers,1,1,1.0


In [18]:
example_cluster = sample_df["C_id_agg"].value_counts().index[4]

sample_df[sample_df["C_id_agg"] == example_cluster][["Brand", "ProductName", LABEL_COL]].head(100)

,Brand,ProductName,Level3
78,Philips,Bluetooth® Hi-Fi adapter AEA2500/12,AV Extenders
1675,ZAGG,IFZ-AU-AX-RED,Audio Cables
2873,Philips,LFH9034,Cable Interface/Gender Adapters
6781,Philips,docking speaker with Bluetooth® AD530/12,Docking Speakers
6785,Philips,docking speaker AD333/05,Docking Speakers
6786,Philips,Micro music system DCM3155/12,Docking Speakers
6787,V7,SP5500-BT-WHT-9NC,Docking Speakers
6788,Philips,wireless speaker dock AS860/10,Docking Speakers
6790,Samsung,ESP-70,Docking Speakers
6791,Philips,docking speaker with Bluetooth® DS7530/12,Docking Speakers


In [19]:
example_cluster = sample_df["C_id_agg"].value_counts().index[14]

sample_df[sample_df["C_id_agg"] == example_cluster][["Brand", "ProductName", LABEL_COL]].head(1000)

,Brand,ProductName,Level3
8116,Chief,HB71P,Flat Panel Ceiling Mounts
8159,Vogel's,VPRO ceiling mount,Flat Panel Ceiling Mounts
8160,Chief,RPA034,Flat Panel Ceiling Mounts
8161,Vogel's,PFA 9015 B,Flat Panel Ceiling Mounts
8162,Chief,RPA181,Flat Panel Ceiling Mounts
8180,Vogel's,VPRO PFA9020 ceiling mount,Flat Panel Ceiling Mounts
8189,Sony,Mounting coupling for pendant applications SNCA-CEILING,Flat Panel Ceiling Mounts
8206,Chief,KITPF018024,Flat Panel Ceiling Mounts
8215,Chief,HB93C,Flat Panel Ceiling Mounts
8226,Newstar,FPMA-C100,Flat Panel Ceiling Mounts


In [21]:
def cluster_quality(df, cluster_col, gold_col):
    stats = (
        df.groupby(cluster_col)[gold_col]
        .value_counts()
        .rename("count")
        .reset_index()
    )

    sizes = df[cluster_col].value_counts().rename("size").reset_index()
    merged = stats.merge(sizes, on=cluster_col)

    # purity
    top = (
        merged.sort_values("count", ascending=False)
        .groupby(cluster_col)
        .first()
        .reset_index()
    )
    top["purity"] = top["count"] / top["size"]
    top["impurity"] = 1 - top["purity"]

    # entropy
    entropy = (
        merged.assign(p=lambda x: x["count"] / x["size"])
        .groupby(cluster_col)
        .apply(lambda g: -np.sum(g["p"] * np.log(g["p"])))
        .reset_index(name="entropy")
    )

    return top.merge(entropy, on=cluster_col).sort_values("impurity", ascending=False)


In [22]:
quality_df = cluster_quality(sample_df, "C_id_agg", LABEL_COL)

# Worst clusters by impurity
quality_df.head(1000)

/var/folders/b_/cv34fjz13m34nzs9_x_gpp600000gn/T/ipykernel_63772/315832772.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: -np.sum(g["p"] * np.log(g["p"])))


,C_id_agg,Level3,count,size,purity,impurity,entropy
157,157,Server Barebones,499,3783,0.131906,0.868094,2.215587
362,362,Audio Cables,2,12,0.166667,0.833333,2.253858
57,57,Audio Cables,29,143,0.202797,0.797203,2.214439
36,36,HDMI Cables,413,1861,0.221924,0.778076,2.133319
83,83,Signage Displays,499,2220,0.224775,0.775225,1.655181
132,132,Battery Chargers,2,8,0.250000,0.750000,1.386294
296,296,Flat Panel Mount Accessories,13,49,0.265306,0.734694,1.727283
7,7,Flat Panel Desk Mounts,454,1703,0.266588,0.733412,1.695108
370,370,USB Cables,24,86,0.279070,0.720930,2.229531
34,34,Portable Speakers,404,1438,0.280946,0.719054,1.803134


In [23]:
example_cluster = sample_df["C_id_agg"].value_counts().index[49]

sample_df[sample_df["C_id_agg"] == example_cluster][["Brand", "ProductName", LABEL_COL]].head(1000)

,Brand,ProductName,Level3
2495,StarTech.com,Micro SATA to SATA Adapter Cable with Power,Cable Interface/Gender Adapters
2534,Techly,USB3.0 to IDE + SATA II Converter,Cable Interface/Gender Adapters
2555,ACT,"Micro SATA(6+7) male - SATA(7) + 5,25"" male (4 pins)Micro SATA(6+7) male - SATA(7) + 5,25"" male (4 pins)",Cable Interface/Gender Adapters
2573,Equip,USB 2.0 SATA/IDE,Cable Interface/Gender Adapters
2578,Techly,USB 3.0 Adapter to Serial ATA IUSB3-SATA2,Cable Interface/Gender Adapters
2580,Conceptronic,SATA to USB 3.0 Adapter,Cable Interface/Gender Adapters
2665,Sandberg,"Adapter kit 2.5"" HDD to 3.5""",Cable Interface/Gender Adapters
2757,j5 create,JEE254,Cable Interface/Gender Adapters
2759,StarTech.com,USB 3.1 (10Gbps) Adapter Cable for 2.5”/3.5” SATA Drives - USB-C,Cable Interface/Gender Adapters
2822,StarTech.com,3 ft SuperSpeed USB 3.0 to eSATA Cable Adapter,Cable Interface/Gender Adapters


In [24]:
sample_df[sample_df["C_id_agg"] == 49][["Brand", "ProductName", LABEL_COL]].head(1000)

,Brand,ProductName,Level3
2861,Zyxel,Telco-50 to RJ-11 Cable,Cable Interface/Gender Adapters
3481,Zyxel,"Antenna cable, type N - type N, 12m",Coaxial Cables
3505,Zyxel,"Antenna cable, type N - SMA, 9m",Coaxial Cables
3545,Zyxel,"Antenna cable, type N - type N, 9m",Coaxial Cables
3577,Zyxel,"Antenna cable, type N - SMA, 3m",Coaxial Cables
3585,Zyxel,LMR-200 Antenna cable 9 m,Coaxial Cables
3587,Zyxel,"Antenna cable, type N - type N, 6m",Coaxial Cables
3613,Zyxel,LMR-400 Antenna cable 9 m,Coaxial Cables
21218,Zyxel,ANT1310,Network Antennas
21219,Zyxel,EXT 106,Network Antennas


In [25]:
sample_df[sample_df["C_id_agg"] == 49][["Brand", "ProductName", LABEL_COL]].nunique()

Brand           14
ProductName    117
Level3           4
dtype: int64

In [40]:
sample_df[sample_df["C_id_agg"] == 5][["Brand", "ProductName", LABEL_COL]].nunique()

Brand            7
ProductName    242
Level3           4
dtype: int64

In [51]:
sample_df[sample_df["C_id_agg"] == 200][["Brand", "ProductName", LABEL_COL]].head(100)

,Brand,ProductName,Level3
1921,Casio,MS-20NC,Calculators
1922,Sharp,EL-243S,Calculators
1923,Canon,F-604,Calculators
1924,HP,12c,Calculators
1925,Sharp,EL-339HB,Calculators
1926,Casio,SL-100NC,Calculators
1927,Casio,JW-200TV-RD,Calculators
1928,Sharp,EL-531XGVL,Calculators
1929,Sharp,EL-334FB,Calculators
1930,HP,10bII+,Calculators


In [ ]:
sample_df["Level3"].head(2)